In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, ArrayType, StringType, DoubleType

Load Bronze Delta Tables

In [0]:
df_fc_bronze = spark.read.table("heliosgrid_catalog.bronze.raw_forecast_telemetry")
df_aq_bronze = spark.read.table("heliosgrid_catalog.bronze.raw_air_quality_telemetry")

In [0]:
display(df_fc_bronze.printSchema())
display(df_aq_bronze.printSchema())

root
 |-- elevation: string (nullable = true)
 |-- generationtime_ms: string (nullable = true)
 |-- hourly: string (nullable = true)
 |-- hourly_units: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- timezone_abbreviation: string (nullable = true)
 |-- utc_offset_seconds: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

root
 |-- elevation: string (nullable = true)
 |-- generationtime_ms: string (nullable = true)
 |-- hourly: string (nullable = true)
 |-- hourly_units: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- timezone_abbreviation: string (nullable = true)
 |-- utc_offset_seconds: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ing

Unnest Parallel Arrays (arrays_zip + explode) & Type Casting & Standard Column Naming

In [0]:
fc_hourly_schema = StructType([
    StructField("time", ArrayType(StringType()), True),
    StructField("shortwave_radiation", ArrayType(DoubleType()), True),
    StructField("direct_normal_irradiance", ArrayType(DoubleType()), True),
    StructField("diffuse_radiation", ArrayType(DoubleType()), True),
    StructField("temperature_2m", ArrayType(DoubleType()), True),
    StructField("relative_humidity_2m", ArrayType(DoubleType()), True),
    StructField("cloud_cover", ArrayType(DoubleType()), True),
    StructField("wind_speed_10m", ArrayType(DoubleType()), True)
])

# Parse string to struct if needed:
if isinstance(df_fc_bronze.schema["hourly"].dataType, StringType):
    df_fc_bronze = df_fc_bronze.withColumn("hourly", F.from_json(F.col("hourly"), fc_hourly_schema))

In [0]:
df_fc_zipped = df_fc_bronze.withColumn(
    "hourly_packed", 
    F.arrays_zip(
        "hourly.time", 
        "hourly.shortwave_radiation", 
        "hourly.direct_normal_irradiance",
        "hourly.diffuse_radiation",
        "hourly.temperature_2m",
        "hourly.relative_humidity_2m",
        "hourly.cloud_cover",
        "hourly.wind_speed_10m"
    )
)

In [0]:
df_fc_silver_rows = df_fc_zipped.select(
    F.col("latitude"),
    F.col("longitude"),
    F.col("elevation"),
    F.col("timezone"),
    F.col("_source_file"),
    F.col("_ingested_timestamp"),
    F.explode("hourly_packed").alias("h")
).select(
    F.col("latitude").cast("double"),
    F.col("longitude").cast("double"),
    F.col("elevation"),
    F.to_timestamp(F.col("h.time")).alias("record_timestamp"),
    F.col("h.shortwave_radiation").cast("double").alias("ghi_wm2"),
    F.col("h.direct_normal_irradiance").cast("double").alias("dni_wm2"),
    F.col("h.diffuse_radiation").cast("double").alias("dhi_wm2"),
    F.col("h.temperature_2m").cast("double").alias("temp_celsius"),
    F.col("h.relative_humidity_2m").cast("double").alias("humidity_pct"),
    F.col("h.cloud_cover").cast("double").alias("cloud_cover_pct"),
    F.col("h.wind_speed_10m").cast("double").alias("wind_speed_ms")
)

In [0]:
aq_hourly_schema = StructType([
    StructField("time", ArrayType(StringType()), True),
    StructField("aerosol_optical_depth", ArrayType(DoubleType()), True),
    StructField("pm2_5", ArrayType(DoubleType()), True),
    StructField("pm10", ArrayType(DoubleType()), True),
    StructField("dust", ArrayType(DoubleType()), True)
])

# Parse string to struct if needed:
if isinstance(df_aq_bronze.schema["hourly"].dataType, StringType):
    df_aq_bronze = df_aq_bronze.withColumn("hourly", F.from_json(F.col("hourly"), aq_hourly_schema))

In [0]:
df_aq_zipped = df_aq_bronze.withColumn(
    "hourly_packed", 
    F.arrays_zip(
        "hourly.time", 
        "hourly.aerosol_optical_depth", 
        "hourly.pm2_5",
        "hourly.pm10",
        "hourly.dust"
    )
)

In [0]:
df_aq_silver_rows = df_aq_zipped.select(
    F.col("latitude"),
    F.col("longitude"),
    F.col("elevation"),
    F.col("timezone"),
    F.col("_source_file"),
    F.col("_ingested_timestamp"),
    F.explode("hourly_packed").alias("h")
).select(
    F.col("latitude").cast("double"),
    F.col("longitude").cast("double"),
    F.col("elevation"),
    F.to_timestamp(F.col("h.time")).alias("record_timestamp"),
    F.col("h.aerosol_optical_depth").cast("double").alias("aod"),
    F.col("h.pm2_5").cast("double").alias("pm2_5"),
    F.col("h.pm10").cast("double").alias("pm10"),
    F.col("h.dust").cast("double").alias("dust")
)

Map Spatial Coordinates to Station Names

In [0]:
df_fc_silver_enrich = df_fc_silver_rows.withColumn(
    "station_id", 
    F.when(F.col("latitude").between(27.0, 28.0), "STN_IND_BHADLA_01")
     .when(F.col("latitude").between(14.0, 14.5), "STN_IND_PAVAGADA_02")
     .when(F.col("latitude").between(23.5, 24.5), "STN_IND_CHARANKA_03")
     .when(F.col("latitude").between(28.0, 29.0), "STN_IND_DELHI_04")
     .otherwise("NA")
).withColumn(
    "station_name",
    F.when(F.col("latitude").between(27.0, 28.0), "Bhadla Solar Park")
     .when(F.col("latitude").between(14.0, 14.5), "Pavagada Solar Park")
     .when(F.col("latitude").between(23.5, 24.5), "Charanka Solar Park")
     .when(F.col("latitude").between(28.0, 29.0), "Indo-Gangetic Station")
     .otherwise("NA")
).withColumn(
    "state",
    F.when(F.col("latitude").between(27.0, 28.0), "Rajasthan")
     .when(F.col("latitude").between(14.0, 14.5), "Karnataka")
     .when(F.col("latitude").between(23.5, 24.5), "Gujarat")
     .when(F.col("latitude").between(28.0, 29.0), "Delhi NCR")
     .otherwise("NA")
).withColumn(
    "climate_zone",
    F.when(F.col("latitude").between(27.0, 28.0), "Arid / Desert")
     .when(F.col("latitude").between(14.0, 14.5), "Peninsular Plateau")
     .when(F.col("latitude").between(23.5, 24.5), "Semi-Arid / Coastal")
     .when(F.col("latitude").between(28.0, 29.0), "Indo-Gangetic Plains")
     .otherwise("NA")
)


In [0]:
df_aq_silver_enrich = df_aq_silver_rows.withColumn(
    "station_id", 
    F.when(F.col("latitude").between(27.0, 28.0), "STN_IND_BHADLA_01")
     .when(F.col("latitude").between(14.0, 14.5), "STN_IND_PAVAGADA_02")
     .when(F.col("latitude").between(23.5, 24.5), "STN_IND_CHARANKA_03")
     .when(F.col("latitude").between(28.0, 29.0), "STN_IND_DELHI_04")
     .otherwise("NA")
)

In [0]:
df_aq_silver_enrich = df_aq_silver_enrich.select(['record_timestamp', 'aod', 'pm2_5', 'pm10', 'dust', 'station_id'])

Join Forecast & Air Quality DataFrames

In [0]:
joined_df = df_fc_silver_enrich.join(df_aq_silver_enrich, ["record_timestamp", "station_id"], "inner")

Data Cleaning & Quality Assertions

In [0]:
df_cleaned = joined_df.withColumn("ghi_wm2", F.when(F.col("ghi_wm2") < 0, 0.0).otherwise(F.col("ghi_wm2"))) \
    .withColumn("temp_celsius", F.when((F.col("temp_celsius") >= -10) & (F.col("temp_celsius") <= 55), F.col("temp_celsius")).otherwise(None)) \
    .withColumn("humidity_pct", F.when((F.col("humidity_pct") >= 0) & (F.col("humidity_pct") <= 100), F.col("humidity_pct")).otherwise(None)) \
    .withColumn("cloud_cover_pct", F.when((F.col("cloud_cover_pct") >= 0) & (F.col("cloud_cover_pct") <= 100), F.col("cloud_cover_pct")).otherwise(None)) \
    .withColumn("wind_speed_ms", F.when((F.col("wind_speed_ms") >= 0) & (F.col("wind_speed_ms") <= 100), F.col("wind_speed_ms")).otherwise(None)) \
    .dropDuplicates(["station_id", "record_timestamp"])

Domain Physics & Indian Monsoon Enrichment

In [0]:
df_enriched = df_cleaned.withColumn("cell_temp_celsius", F.col("temp_celsius") + ((25.0 / 800.0) * F.col("ghi_wm2"))) \
    .withColumn("thermal_derating_factor", F.when(F.col("cell_temp_celsius") > 25.0, \
        1.0 - (0.004 * (F.col("cell_temp_celsius") - 25.0))).otherwise(1.0)) \
    .withColumn("monsoon_season",
        F.when(F.month(F.col("record_timestamp")).between(3, 5), "Pre-Monsoon")
         .when(F.month(F.col("record_timestamp")).between(6, 9), "Southwest Monsoon")
         .when(F.month(F.col("record_timestamp")).between(10, 12), "Northeast Monsoon")
         .otherwise("Winter / Smog Season"))

In [0]:
final_df = df_enriched.select(['record_timestamp', 'station_id', 'station_name', 'state', 'climate_zone', 'latitude', 'longitude', 'elevation', 'ghi_wm2', 'dni_wm2', 'dhi_wm2', 'temp_celsius', 'humidity_pct', 'cloud_cover_pct', 'wind_speed_ms', 'aod', 'pm2_5', 'pm10', 'dust', 'cell_temp_celsius', 'thermal_derating_factor', 'monsoon_season'])

 Write to Silver Delta Table & Z-Order

In [0]:
final_df.write.mode("append") \
    .option("path", "s3://heliosgrid/catalog/silver/tables/cleaned_solar_telemetry/") \
    .saveAsTable("heliosgrid_catalog.silver.cleaned_solar_telemetry")

In [0]:
%sql
OPTIMIZE heliosgrid_catalog.silver.cleaned_solar_telemetry
ZORDER BY (record_timestamp, station_id);